## Câu 31

Viết thuật toán cho phương pháp lặp đơn giải hệ phương trình $X = BX + d$ với sai số $\varepsilon$ cho trước.

## Phương pháp lặp đơn

### Bài toán

Cho hệ phương trình:

$$x = Bx + d$$

với $B \in \text{mat}(n, \mathbb{R})$, $d \in \mathbb{R}^n$, và sai số cho trước $\varepsilon > 0$.

**Yêu cầu:** Tìm nghiệm xấp xỉ $x^* \in \mathbb{R}^n$ thoả $\|x^{(k+1)} - x^{(k)}\| < \varepsilon$.

---

### Điều kiện hội tụ

Tính chuẩn ma trận $B$ theo chuẩn hàng:

$$q = \|B\|_\infty = \max_{1 \leq i \leq n} \sum_{j=1}^{n} |b_{ij}|$$

Nếu $q < 1$, phương pháp lặp hội tụ với mọi vector khởi đầu $x^{(0)}$.

---

### Thuật toán

**Đầu vào:** Ma trận $B$, vector $d$, sai số $\varepsilon > 0$, số vòng lặp tối đa $N$

**Đầu ra:** Nghiệm xấp xỉ $x^*$ hoặc thông báo không hội tụ

---

**Bước 1 — Kiểm tra điều kiện hội tụ**

&emsp;Tính $q = \|B\|_\infty$

&emsp;Nếu $q \geq 1$: thông báo *"Không đảm bảo hội tụ"* và dừng

**Bước 2 — Khởi tạo**

&emsp;Chọn $x^{(0)} \in \mathbb{R}^n$ (thường lấy $x^{(0)} = \mathbf{0}$ hoặc $x^{(0)} = d$)

&emsp;Đặt $k \leftarrow 0$

**Bước 3 — Vòng lặp:** lặp khi $k < N$

&emsp;**3.1.** Tính $x^{(k+1)} = B \cdot x^{(k)} + d$

&emsp;**3.2.** Tính sai số theo một trong hai cách:

| | Công thức | Ý nghĩa |
|---|---|---|
| **Hậu nghiệm** | $\delta = \dfrac{q}{1-q}\|x^{(k+1)} - x^{(k)}\|_\infty$ | Ước lượng từ hai bước lặp liên tiếp |
| **Tiên nghiệm** | $\delta = \dfrac{q^k}{1-q}\|x^{(1)} - x^{(0)}\|_\infty$ | Ước lượng từ bước đầu, tính trước vòng lặp |

&emsp;**3.3.** Nếu $\delta < \varepsilon$: **dừng**, trả về $x^{(k+1)}$

&emsp;**3.4.** Đặt $k \leftarrow k + 1$

**Bước 4 — Kết thúc**

&emsp;Nếu $k = N$ mà chưa hội tụ: thông báo *"Vượt quá số vòng lặp tối đa"*

---

### Lưu ý

- **Chuẩn vector:** Thường dùng $\|\cdot\|_\infty = \max_i |x_i|$ vì tính toán đơn giản và tương thích với $\|B\|_\infty$.

- **Sai số hậu nghiệm vs tiên nghiệm:** Sai số hậu nghiệm chính xác hơn nhưng tốn thêm một phép tính chuẩn mỗi vòng lặp. Sai số tiên nghiệm cho phép ước lượng số vòng lặp cần thiết *trước khi* bắt đầu lặp theo công thức:

$$k \geq \left\lceil \frac{\ln(\varepsilon(1-q)) - \ln\|x^{(1)} - x^{(0)}\|_\infty}{\ln q} \right\rceil$$

- **Chọn $x^{(0)}$:** Khi $q$ gần 1 (hội tụ chậm), chọn $x^{(0)}$ gần nghiệm thực sẽ giảm đáng kể số vòng lặp.

- **Điều kiện $q \geq 1$:** Chỉ là điều kiện đủ. Nếu $q \geq 1$ nhưng bán kính phổ $\rho(B) < 1$, phương pháp vẫn có thể hội tụ — cần kiểm tra $\rho(B)$ trực tiếp.

In [ ]:
import numpy as np


def lap_don(B, d, eps=1e-6, max_iter=1000, log=4):
    """
    Phương pháp lặp đơn: x = Bx + d

    Tham số:
        B       : ma trận lặp (n x n)
        d       : vector hằng (n,)
        eps     : sai số cho phép
        max_iter: số vòng lặp tối đa
        log     : in ra log lần lặp đầu tiên (0 = tắt)

    Trả về:
        x    : nghiệm xấp xỉ
        k    : số vòng lặp đã thực hiện
        info : "ok" hoặc thông báo lỗi
    """
    B, d = np.array(B, dtype=float), np.array(d, dtype=float)

    # Kiểm tra hội tụ
    q = np.max(np.sum(np.abs(B), axis=1))
    if q >= 1:
        return None, 0, f"Không đảm bảo hội tụ (q = {q:.4f} >= 1)"

    x = d.copy()
    if log:
        print(f"{'k':>3}  {'x':>40}  {'sai số':>12}")
        print(f"{'0':>3}  {np.array2string(x, precision=6):>40}")

    for k in range(1, max_iter + 1):
        x_new = B @ x + d
        delta = q / (1 - q) * np.max(np.abs(x_new - x))
        x = x_new

        if log and k <= log:
            print(f"{k:>3}  {np.array2string(x, precision=6):>40}  {delta:>12.2e}")

        if delta < eps:
            return x, k, "ok"

    return x, max_iter, "Vượt quá số vòng lặp tối đa"


# ====== Ví dụ ======
if __name__ == "__main__":
    # Hệ x = Bx + d
    B = [[0.0, 0.1, 0.2],
         [0.1, 0.0, 0.1],
         [0.2, 0.1, 0.0]]

    d = [1.0, 2.0, 3.0]



    x, k, info = lap_don(B, d, eps=1e-6)
    print()
    print(f"Trạng thái : {info}")
    print(f"Số vòng lặp: {k}")
    print(f"Nghiệm     : {x}")

  k                                         x        sai số
  0                                [1. 2. 3.]
  1                             [1.8 2.4 3.4]      3.43e-01
  2                          [1.92 2.52 3.6 ]      8.57e-02
  3                       [1.972 2.552 3.636]      2.23e-02
  4                    [1.9824 2.5608 3.6496]      5.83e-03

Trạng thái : ok
Số vòng lặp: 11
Nghiệm     : [1.98717898 2.56410219 3.65384564]


## Câu 32

Viết thuật toán cho phương pháp lặp Jacobi giải hệ phương trình $AX = B$ với sai số $\varepsilon$ cho trước.

## Phương pháp lặp Jacobi

### Bài toán

Cho hệ phương trình tuyến tính:

$$Ax = b$$

với $A \in \text{mat}(n, \mathbb{R})$ khả nghịch, $b \in \mathbb{R}^n$, sai số cho trước $\varepsilon > 0$.

**Yêu cầu:** Tìm nghiệm xấp xỉ $x^* \in \mathbb{R}^n$ bằng cách tách $A = D - (L + U)$, trong đó:

- $D$ — ma trận đường chéo của $A$
- $L$ — phần tam giác dưới nghiêm ngặt (không gồm đường chéo)
- $U$ — phần tam giác trên nghiêm ngặt (không gồm đường chéo)

Hệ được viết lại thành dạng lặp:

$$x = Bx + d, \quad B = D^{-1}(L + U), \quad d = D^{-1}b$$

Hay ở dạng thành phần (dễ cài đặt hơn):

$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j \neq i} a_{ij} x_j^{(k)} \right), \quad i = 1, 2, \ldots, n$$

---

### Điều kiện hội tụ

Tính chuẩn ma trận lặp $B = D^{-1}(L+U)$ theo chuẩn hàng:

$$q = \|B\|_\infty = \max_{1 \leq i \leq n} \frac{1}{|a_{ii}|} \sum_{j \neq i} |a_{ij}|$$

Nếu $q < 1$, phương pháp hội tụ với mọi vector khởi đầu $x^{(0)}$.

**Điều kiện đủ thực tế:** $A$ là ma trận **chéo trội nghiêm ngặt**, tức là:

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}|, \quad \forall i = 1, \ldots, n$$

---

### Thuật toán

**Đầu vào:** Ma trận $A$, vector $b$, sai số $\varepsilon > 0$, số vòng lặp tối đa $N$

**Đầu ra:** Nghiệm xấp xỉ $x^*$ hoặc thông báo không hội tụ

---

**Bước 1 — Kiểm tra điều kiện hội tụ**

&emsp;Tính $q = \|B\|_\infty = \max_{i} \dfrac{1}{|a_{ii}|} \sum_{j \neq i} |a_{ij}|$

&emsp;Nếu $q \geq 1$: thông báo *"Không đảm bảo hội tụ"* và dừng

**Bước 2 — Khởi tạo**

&emsp;Chọn $x^{(0)} \in \mathbb{R}^n$ (thường lấy $x^{(0)} = \mathbf{0}$)

&emsp;Đặt $k \leftarrow 0$

**Bước 3 — Vòng lặp:** lặp khi $k < N$

&emsp;**3.1.** Với mỗi $i = 1, \ldots, n$, tính:

$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=1}^{i-1} a_{ij} x_j^{(k)} - \sum_{j=i+1}^{n} a_{ij} x_j^{(k)} \right)$$

&emsp;*(Lưu ý: Jacobi dùng toàn bộ $x^{(k)}$ cũ để tính $x^{(k+1)}$, khác với Gauss-Seidel)*

&emsp;**3.2.** Tính sai số theo một trong hai cách:
| | Công thức | Ý nghĩa |
|---|---|---|
| **Hậu nghiệm** | $\delta = \dfrac{q}{1-q}\|x^{(k+1)} - x^{(k)}\|_\infty$ | Ước lượng từ hai bước lặp liên tiếp |
| **Tiên nghiệm** | $\delta = \dfrac{q^k}{1-q}\|x^{(1)} - x^{(0)}\|_\infty$ | Ước lượng từ bước đầu, tính trước vòng lặp |

&emsp;**3.3.** Nếu $\delta < \varepsilon$: **dừng**, trả về $x^{(k+1)}$

&emsp;**3.4.** Đặt $k \leftarrow k + 1$

**Bước 4 — Kết thúc**

&emsp;Nếu $k = N$ mà chưa hội tụ: thông báo *"Vượt quá số vòng lặp tối đa"*

---

### Lưu ý

- **Jacobi vs Gauss-Seidel:** Jacobi tính toàn bộ $x^{(k+1)}$ từ $x^{(k)}$ cũ — các thành phần độc lập nhau nên **dễ song song hóa**. Gauss-Seidel dùng ngay $x_j^{(k+1)}$ vừa tính được, thường hội tụ nhanh hơn nhưng tuần tự.

- **Số vòng lặp tiên nghiệm:** Có thể ước lượng trước số bước cần thiết:

$$k \geq \left\lceil \frac{\ln(\varepsilon(1-q)) - \ln\|x^{(1)} - x^{(0)}\|_\infty}{\ln q} \right\rceil$$

- **Yêu cầu $a_{ii} \neq 0$:** Nếu có phần tử đường chéo bằng 0, cần hoán vị hàng trước khi áp dụng Jacobi.

- **Điều kiện $q \geq 1$:** Chỉ là điều kiện đủ. Nếu $q \geq 1$ nhưng bán kính phổ $\rho(B) < 1$, phương pháp vẫn có thể hội tụ — cần kiểm tra $\rho(B)$ trực tiếp.

In [3]:
import numpy as np


def jacobi(A, b, eps=1e-6, max_iter=1000, log=4):
    """
    Phương pháp lặp Jacobi: Ax = b

    Tham số:
        A       : ma trận hệ số (n x n)
        b       : vector vế phải (n,)
        eps     : sai số cho phép
        max_iter: số vòng lặp tối đa
        log     : in ra log lần lặp đầu tiên (0 = tắt)

    Trả về:
        x    : nghiệm xấp xỉ
        k    : số vòng lặp đã thực hiện
        info : "ok" hoặc thông báo lỗi
    """
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    n = len(b)
    diag = np.diag(A)

    if np.any(diag == 0):
        return None, 0, "Phần tử đường chéo = 0, cần hoán vị hàng"

    # Tính ma trận lặp B = D^{-1}(L+U) và kiểm tra hội tụ
    D_inv = np.diag(1.0 / diag)
    B = D_inv @ (np.diag(diag) - A)
    q = np.max(np.sum(np.abs(B), axis=1))
    if q >= 1:
        return None, 0, f"Không đảm bảo hội tụ (q = {q:.4f} >= 1)"

    x = np.zeros(n)
    if log:
        print(f"{'k':>3}  {'x':>40}  {'sai số':>12}")
        print(f"{'0':>3}  {np.array2string(x, precision=6):>40}")

    for k in range(1, max_iter + 1):
        x_new = np.empty(n)
        for i in range(n):
            x_new[i] = (b[i] - A[i] @ x + A[i, i] * x[i]) / A[i, i]

        delta = q / (1 - q) * np.max(np.abs(x_new - x))
        x = x_new

        if log and k <= log:
            print(f"{k:>3}  {np.array2string(x, precision=6):>40}  {delta:>12.2e}")

        if delta < eps:
            return x, k, "ok"

    return x, max_iter, "Vượt quá số vòng lặp tối đa"


# ====== Ví dụ ======
if __name__ == "__main__":
    # Hệ Ax = b (A chéo trội)
    A = [[10, -1,  2],
         [-1, 11, -1],
         [ 2, -1, 10]]

    b = [6, 25, -11]

    x, k, info = jacobi(A, b, eps=1e-6)
    print()
    print(f"Trạng thái : {info}")
    print(f"Số vòng lặp: {k}")
    print(f"Nghiệm     : {x}")

  k                                         x        sai số
  0                                [0. 0. 0.]
  1           [ 0.6       2.272727 -1.1     ]      9.74e-01
  2           [ 1.047273  2.227273 -0.992727]      1.92e-01
  3           [ 1.021273  2.277686 -1.086727]      4.03e-02
  4           [ 1.045114  2.266777 -1.076486]      1.02e-02

Trạng thái : ok
Số vòng lặp: 11
Nghiệm     : [ 1.04326886  2.26923101 -1.0817311 ]


## Câu 33

Nền kinh tế gồm 10 ngành chính: (1) bất động sản, (2) máy móc công nghiệp, (3) khai khoáng, (4) nông nghiệp, (5) thủy sản, (6) may mặc, (7) năng lượng, (8) dịch vụ, (9) giải trí, (10) đồ gia dụng. Ma trận tiêu thụ nội bộ (thể hiện lượng sản phẩm cần thiết để sản xuất một đơn vị sản phẩm) và vector nhu cầu bên ngoài (mức nhu cầu tiêu thụ sản phẩm ngoài thị trường của 10 ngành trên) cho trong file **CK20222** kèm với **pass mở là `20svds222`**.

Tính lượng sản phẩm cần sản xuất để đáp ứng nhu cầu trên (đơn vị tính theo triệu đô). Cần nêu rõ phương pháp, thuật toán bạn dùng để tìm kiếm. _(Cần ghi lại một số kết quả trung gian mà bạn nghĩ là cần thiết đối với phương pháp đã lựa chọn.)_

In [1]:
import numpy as np

# ── Hiển thị ma trận / vector ────────────────────────────────────
def _print_mat(M, label):
    n, m = M.shape
    sep = "─" * (12 * m)
    print(f"\n{label}:\n{sep}")
    for row in M:
        print("  " + "  ".join(f"{v:10.4f}" for v in row))
    print(sep)

def _print_vec(v, label):
    print(f"\n{label}:")
    for i, val in enumerate(v):
        print(f"  x{i+1} = {val:10.4f}")


# ── Phân tách LU (Doolittle) ─────────────────────────────────────
def lu_decompose(A):
    n = len(A)
    L, U = np.eye(n), np.zeros((n, n))
    print("=== Bước 1: Phân tách LU ===")
    for k in range(n):
        U[k, k:] = A[k, k:] - L[k, :k] @ U[:k, k:]
        if abs(U[k, k]) < 1e-12:
            raise ValueError(f"Ma trận suy biến tại cột {k+1}")
        L[k+1:, k] = (A[k+1:, k] - L[k+1:, :k] @ U[:k, k]) / U[k, k]
        print(f"  k={k+1}: U[{k+1},{k+1}]={U[k,k]:.4f}", end="")
        if k < n-1:
            print(f"  |  L[{k+2}..{n},{k+1}] =", "  ".join(f"{L[i,k]:.4f}" for i in range(k+1, n)))
        else:
            print()
    _print_mat(L, "L"); _print_mat(U, "U")
    print(f"\n  L @ U == A : {np.allclose(L @ U, A)}")
    return L, U


# ── Thế xuôi LY = B ──────────────────────────────────────────────
def forward_sub(L, b):
    n = len(b)
    Y = np.zeros(n)
    print("\n=== Bước 2: Thế xuôi LY = B ===")
    for i in range(n):
        Y[i] = b[i] - L[i, :i] @ Y[:i]
        print(f"  y{i+1} = {Y[i]:.4f}")
    return Y


# ── Thế ngược UX = Y ─────────────────────────────────────────────
def backward_sub(U, Y):
    n = len(Y)
    X = np.zeros(n)
    print("\n=== Bước 3: Thế ngược UX = Y ===")
    for i in range(n-1, -1, -1):
        X[i] = (Y[i] - U[i, i+1:] @ X[i+1:]) / U[i, i]
        print(f"  x{i+1} = {X[i]:.4f}")
    return X


# ── Hàm tổng hợp ─────────────────────────────────────────────────
def lu_solve(A, b):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    L, U = lu_decompose(A)
    Y    = forward_sub(L, b)
    X    = backward_sub(U, Y)
    err  = np.max(np.abs(A @ X - b))
    print(f"\n  ||A·X - B||_∞ = {err:.2e}  →  {'ĐÚNG ✓' if err < 1e-8 else 'SAI ✗'}")
    return X


# ── Dữ liệu bài toán cấp 7 ───────────────────────────────────────
A = np.array([
    [0.05953,	0.03186,	0.09466,	0.09514,	0.03478,	0.047,	0.01516,	0.09841,	0.01701,	0.03281],
    [0.06035,	0.07194,	0.09583,	0.08516,	0.00064,	0.05308,	0.08699,	0.06794,	0.04468,	0.06082],
    [0.07307,	0.08444,	0.06385,	0.07472,	0.04107,	0.08514,	0.07277,	0.09001,	0.07051,	0.03304],
    [0.07984,	0.04551,	0.05665,	0.04349,	0.05256,	0.01644,	0.0891,	0.01881,	0.0014,	0.06476],
    [0.05603,	0.0098,	0.07562,	0.04857,	0.01058,	0.03132,	0.06842,	0.04374,	0.00275,	0.09992],
    [0.05876,	0.05356,	0.01616,	0.01209,	0.00999,	0.07356,	0.02366,	0.05048,	0.08056,	0.0216],
    [0.07854,	0.0897,	0.03872,	0.03786,	0.03373,	0.00599,	0.08687,	0.04372,	0.05682,	0.0629],
    [0.07909,	0.03647,	0.09861,	0.04025,	0.03746,	0.00255,	0.04359,	0.06612,	0.04205,	0.02439],
    [0.02533,	0.09595,	0.00065,	0.04671,	0.03464,	0.04578,	0.07115,	0.04019,	0.04568,	0.01673],
    [0.00902,	0.09266,	0.09269,	0.07965,	0.05633,	0.00469,	0.00403,	0.07378,	0.04205,	0.06873]
        ],dtype =float)

b = np.array([29852, 5035, 15243, 4198, 25630, 44127, 18750, 32496, 19828, 9846], dtype=float)


X = lu_solve(np.eye(len(A)) - A, b)

=== Bước 1: Phân tách LU ===
  k=1: U[1,1]=0.9405  |  L[2..10,1] = -0.0642  -0.0777  -0.0849  -0.0596  -0.0625  -0.0835  -0.0841  -0.0269  -0.0096
  k=2: U[2,2]=0.9260  |  L[3..10,2] = -0.0939  -0.0521  -0.0126  -0.0600  -0.0997  -0.0423  -0.1045  -0.1004
  k=3: U[3,3]=0.9192  |  L[4..10,3] = -0.0761  -0.0898  -0.0307  -0.0618  -0.1206  -0.0151  -0.1130
  k=4: U[4,4]=0.9368  |  L[5..10,4] = -0.0678  -0.0281  -0.0646  -0.0673  -0.0642  -0.1067
  k=5: U[5,5]=0.9794  |  L[6..10,5] = -0.0157  -0.0444  -0.0508  -0.0412  -0.0697
  k=6: U[6,6]=0.9157  |  L[7..10,6] = -0.0276  -0.0268  -0.0635  -0.0304
  k=7: U[7,7]=0.8867  |  L[8..10,7] = -0.0796  -0.1064  -0.0452
  k=8: U[8,8]=0.8964  |  L[9..10,8] = -0.0772  -0.1203
  k=9: U[9,9]=0.9287  |  L[10..10,9] = -0.0764
  k=10: U[10,10]=0.8893

L:
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      1.0000      0.0000      0.0000      0.0000      0.0000      0.0000      0.000

## Câu 34

Nền kinh tế gồm 7 ngành chính: (1) bất động sản, (2) sản xuất máy móc công nghiệp, (3) khai khoáng, (4) nông nghiệp, (5) năng lượng, (6) dịch vụ, (7) giải trí có ma trận tiêu thụ nội bộ (thể hiện lượng sản phẩm cần thiết để sản xuất một đơn vị sản phẩm) và vector nhu cầu bên ngoài (mức nhu cầu tiêu thụ của các ngành ngoài 7 ngành trên) là:

$$
C = \begin{bmatrix}
.1588 & .0064 & .0025 & .0304 & .0014 & .0083 & .1594 \\
.0057 & .2645 & .0436 & .0099 & .0083 & .0201 & .3413 \\
.0264 & .1506 & .3557 & .0139 & .0142 & .0070 & .0236 \\
.3299 & .0565 & .0495 & .3636 & .0204 & .0483 & .0649 \\
.0089 & .0081 & .0333 & .0295 & .3412 & .0237 & .0020 \\
.1190 & .0901 & .0996 & .1260 & .1722 & .2368 & .3369 \\
.0063 & .0126 & .0196 & .0098 & .0064 & .0132 & .0012
\end{bmatrix},
\qquad
d = \begin{bmatrix} 74000 \\ 56000 \\ 10500 \\ 25000 \\ 17500 \\ 196000 \\ 5000 \end{bmatrix}
$$

(đơn vị tính theo triệu đô). Tính lượng sản phẩm cần sản xuất để đáp ứng nhu cầu trên. Cần nêu rõ phương pháp, thuật toán bạn dùng để tìm kiếm. _(Nếu dùng một phương pháp lặp để giải thì cần ghi rõ số lần lặp, xấp xỉ đầu và 3 xấp xỉ cuối; nếu dùng một phương pháp tính đúng, cần ghi lại một số kết quả trung gian mà bạn nghĩ là quan trọng nhất trong phương pháp.)_

In [8]:
import numpy as np


def lap_don(B, d, eps=1e-6, max_iter=1000, log=10):
    """
    Phương pháp lặp đơn: x = Bx + d

    Tham số:
        B       : ma trận lặp (n x n)
        d       : vector hằng (n,)
        eps     : sai số cho phép
        max_iter: số vòng lặp tối đa
        log     : in ra log lần lặp đầu tiên (0 = tắt)

    Trả về:
        x    : nghiệm xấp xỉ
        k    : số vòng lặp đã thực hiện
        info : "ok" hoặc thông báo lỗi
    """
    B, d = np.array(B, dtype=float), np.array(d, dtype=float)

    # Kiểm tra hội tụ
    # Tính theo chuẩn cột, chỉ cần 1 trong 2 chuẩn này <1 là đủ
    q = np.max(np.sum(np.abs(B), axis=0))
    if q >= 1:
        return None, 0, f"Không đảm bảo hội tụ (q = {q:.4f} >= 1)"

    x = d.copy()
    if log:
        print(f"{'k':>3}  {'x':>40}  {'sai số':>12}")
        print(f"{'0':>3}  {np.array2string(x, precision=6):>40}")

    for k in range(1, max_iter + 1):
        x_new = B @ x + d
        delta = q / (1 - q) * np.max(np.abs(x_new - x))
        x = x_new

        if log and k <= log:
            print(f"{k:>3}  {np.array2string(x, precision=6):>40}  {delta:>12.2e}")

        if delta < eps:
            return x, k, "ok"

    return x, max_iter, "Vượt quá số vòng lặp tối đa"


# ====== Ví dụ ======
if __name__ == "__main__":
    # Hệ x = Bx + d
    B = [[0.1190, 0.0064 , 0.0025 , 0.0304 , 0.0014 , 0.0083 , 0.1594],
        [0.0057 , 0.2645 , 0.0436 , 0.0099 , 0.0083 , 0.0201 , 0.3413], 
        [0.0264 , 0.1506 , 0.3557 , 0.0139 , 0.0142 , 0.0070 , 0.0236],
        [0.3299 , 0.0565 , 0.0495 , 0.3636 , 0.0204 , 0.0483 , 0.0649],
        [0.0089 , 0.0081 , 0.0333 , 0.0295 , 0.3412 , 0.0237 , 0.0020],
        [0.1190 , 0.0901 , 0.0996 , 0.1260 ,  0.1722 , 0.2368 , 0.3369],
        [0.0063, 0.0126 , 0.0196 , 0.0098 , 0.0064 , 0.0132 , 0.0012]]
    d = [74000, 56000, 10500, 25000, 17500, 196000, 5000]


    x, k, info = lap_don(B, d, eps=1e-6)
    print()
    print(f"Trạng thái : {info}")
    print(f"Số vòng lặp: {k}")
    print(f"Nghiệm     : {x}")

  k                                         x        sai số
  0  [ 74000.  56000.  10500.  25000.  17500. 196000.   5000.]
  1  [ 86398.95  77730.45  26708.05  72334.65  30325.55 265158.2    9327.8 ]      9.09e+05
  2  [ 90774.813565  87697.72208   37499.50936   99548.867745  38571.80014
 296213.333565  11461.439525]      4.08e+05
  3  [ 92823.027237  92519.799582  43717.739634 113791.542477  43407.549742
 311628.61639   12558.071162]      2.03e+05
  4  [ 93855.668828  94943.295503  47110.270165 121140.513323  46109.556875
 319573.246005  13128.9351  ]      1.04e+05
  5  [ 94386.672539  96187.81495   48918.837491 124934.02099   47579.497892
 323717.251685  13427.337296]      5.45e+04
  6  [ 94661.689431  96833.766021  49873.241266 126897.866507  48366.791815
 325885.643689  13583.454439]      2.85e+04
  7  [ 94804.622379  97170.64375   50374.603501 127917.314164  48784.514568
 327020.717147  13665.126825]      1.49e+04
  8  [ 94879.056531  97346.671413  50637.420232 128447.636064  4900

In [9]:
import numpy as np

# ── Hiển thị ma trận / vector ────────────────────────────────────
def _print_mat(M, label):
    n, m = M.shape
    sep = "─" * (12 * m)
    print(f"\n{label}:\n{sep}")
    for row in M:
        print("  " + "  ".join(f"{v:10.4f}" for v in row))
    print(sep)

def _print_vec(v, label):
    print(f"\n{label}:")
    for i, val in enumerate(v):
        print(f"  x{i+1} = {val:10.4f}")


# ── Phân tách LU (Doolittle) ─────────────────────────────────────
def lu_decompose(A):
    n = len(A)
    L, U = np.eye(n), np.zeros((n, n))
    print("=== Bước 1: Phân tách LU ===")
    for k in range(n):
        U[k, k:] = A[k, k:] - L[k, :k] @ U[:k, k:]
        if abs(U[k, k]) < 1e-12:
            raise ValueError(f"Ma trận suy biến tại cột {k+1}")
        L[k+1:, k] = (A[k+1:, k] - L[k+1:, :k] @ U[:k, k]) / U[k, k]
        print(f"  k={k+1}: U[{k+1},{k+1}]={U[k,k]:.4f}", end="")
        if k < n-1:
            print(f"  |  L[{k+2}..{n},{k+1}] =", "  ".join(f"{L[i,k]:.4f}" for i in range(k+1, n)))
        else:
            print()
    _print_mat(L, "L"); _print_mat(U, "U")
    print(f"\n  L @ U == A : {np.allclose(L @ U, A)}")
    return L, U


# ── Thế xuôi LY = B ──────────────────────────────────────────────
def forward_sub(L, b):
    n = len(b)
    Y = np.zeros(n)
    print("\n=== Bước 2: Thế xuôi LY = B ===")
    for i in range(n):
        Y[i] = b[i] - L[i, :i] @ Y[:i]
        print(f"  y{i+1} = {Y[i]:.4f}")
    return Y


# ── Thế ngược UX = Y ─────────────────────────────────────────────
def backward_sub(U, Y):
    n = len(Y)
    X = np.zeros(n)
    print("\n=== Bước 3: Thế ngược UX = Y ===")
    for i in range(n-1, -1, -1):
        X[i] = (Y[i] - U[i, i+1:] @ X[i+1:]) / U[i, i]
        print(f"  x{i+1} = {X[i]:.4f}")
    return X


# ── Hàm tổng hợp ─────────────────────────────────────────────────
def lu_solve(A, b):
    A, b = np.array(A, dtype=float), np.array(b, dtype=float)
    L, U = lu_decompose(A)
    Y    = forward_sub(L, b)
    X    = backward_sub(U, Y)
    err  = np.max(np.abs(A @ X - b))
    print(f"\n  ||A·X - B||_∞ = {err:.2e}  →  {'ĐÚNG ✓' if err < 1e-8 else 'SAI ✗'}")
    return X


# ── Dữ liệu bài toán cấp 7 ───────────────────────────────────────
A = [[0.1190, 0.0064 , 0.0025 , 0.0304 , 0.0014 , 0.0083 , 0.1594],
        [0.0057 , 0.2645 , 0.0436 , 0.0099 , 0.0083 , 0.0201 , 0.3413], 
        [0.0264 , 0.1506 , 0.3557 , 0.0139 , 0.0142 , 0.0070 , 0.0236],
        [0.3299 , 0.0565 , 0.0495 , 0.3636 , 0.0204 , 0.0483 , 0.0649],
        [0.0089 , 0.0081 , 0.0333 , 0.0295 , 0.3412 , 0.0237 , 0.0020],
        [0.1190 , 0.0901 , 0.0996 , 0.1260 ,  0.1722 , 0.2368 , 0.3369],
        [0.0063, 0.0126 , 0.0196 , 0.0098 , 0.0064 , 0.0132 , 0.0012]]
b = [74000, 56000, 10500, 25000, 17500, 196000, 5000]



X = lu_solve(np.eye(len(A)) - A, b)

=== Bước 1: Phân tách LU ===
  k=1: U[1,1]=0.8810  |  L[2..7,1] = -0.0065  -0.0300  -0.3745  -0.0101  -0.1351  -0.0072
  k=2: U[2,2]=0.7355  |  L[3..7,2] = -0.2050  -0.0801  -0.0111  -0.1237  -0.0172
  k=3: U[3,3]=0.6353  |  L[4..7,3] = -0.0849  -0.0532  -0.1658  -0.0321
  k=4: U[4,4]=0.6228  |  L[5..7,4] = -0.0495  -0.2154  -0.0172
  k=5: U[5,5]=0.6567  |  L[6..7,5] = -0.2756  -0.0114
  k=6: U[6,6]=0.7385  |  L[7..7,6] = -0.0206
  k=7: U[7,7]=0.9762

L:
────────────────────────────────────────────────────────────────────────────────────
      1.0000      0.0000      0.0000      0.0000      0.0000      0.0000      0.0000
     -0.0065      1.0000      0.0000      0.0000      0.0000      0.0000      0.0000
     -0.0300     -0.2050      1.0000      0.0000      0.0000      0.0000      0.0000
     -0.3745     -0.0801     -0.0849      1.0000      0.0000      0.0000      0.0000
     -0.0101     -0.0111     -0.0532     -0.0495      1.0000      0.0000      0.0000
     -0.1351     -0.1237     -0

## Câu 35

Dùng phương pháp lặp Jacobi tìm nghiệm gần đúng của phương trình $(A + aI)x = b$ với sai số tỉ đối không vượt quá $10^{-5}$, biết $a$ là số thứ tự theo danh sách của bạn; $I$ là ma trận đơn vị cùng cấp với $A$; ma trận $A$, vector $b$ cho trong file **GK20222** với pass mở file là **`gts2023gts`**.

Thực hiện thuật toán theo phương pháp đã chọn để tìm nghiệm theo yêu cầu. Đánh giá sai số tương đối cho nghiệm tìm được.

## Câu 36

Chọn một phương pháp phù hợp để giải hệ phương trình sau. Thực hiện các bước theo thuật toán của phương pháp để giải và đưa ra nghiệm với sai lệch không quá $10^{-5}$. ($a$ là số thứ tự của bạn theo danh sách)

$$
\begin{cases}
(27.5 + a)\,x_1 + 1.7x_2 - 3.2x_3 + 2.1x_4 + 9.23x_5 - 3.52x_6 = 21.41 \\
2.5x_1 + (47.1 + a)\,x_2 + 5.2x_3 + 2.8x_4 + 7.23x_5 - 5.52x_6 = 27.11 \\
11.3x_1 + 2.7x_2 - (38.2 + a)\,x_3 + 4.1x_4 - 7.58x_5 + 4.25x_6 = 14.17 \\
8.4x_1 - 4.6x_2 - 6.5x_3 + (52.1 + a)\,x_4 + 1.43x_5 + 15.26x_6 = 52.49 \\
42.7x_1 - 36.9x_2 - 42.7x_3 + 61.1x_4 + (243 + a)\,x_5 - 35.26x_6 = 56.72 \\
9.2x_1 - x_2 + 35x_3 - 2x_4 + 14.73x_5 + (75.64 + a)\,x_6 = 18.57
\end{cases}
$$